In [ ]:
# set ON_KAGGLE = True before running this notebook on Kaggle; data and output paths follow it
ON_KAGGLE = False

from pathlib import Path

COMPETITION = "hull-tactical-market-prediction"

if ON_KAGGLE:
    # competition data is mounted read-only, under /kaggle/input/competitions/<slug> or /kaggle/input/<slug>
    DATA_DIR = next(p for p in [Path("/kaggle/input/competitions") / COMPETITION,
                                Path("/kaggle/input") / COMPETITION] if p.exists())
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path("data")
    OUT_DIR = Path(".")

print(f"Kaggle: {ON_KAGGLE}, data: {DATA_DIR}, output: {OUT_DIR}")

In [1]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor 

from catboost import CatBoostRegressor
import sys


In [3]:
import pandas.api.types

MIN_INVESTMENT = 0
MAX_INVESTMENT = 2


class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    if not pandas.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution
    solution['position'] = submission['prediction']

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

In [4]:
train_data = pd.read_csv(DATA_DIR / "train.csv").sort_values("date_id").reset_index(drop=True)

# time series: validate on the most recent 20% of days instead of a random split,
# otherwise the model is trained on days that come after the validation days
split = int(len(train_data) * 0.8)
train_part = train_data.iloc[:split]
val_part = train_data.iloc[split:]
print(f"train days: {train_part.date_id.min()}-{train_part.date_id.max()}, "
      f"validation days: {val_part.date_id.min()}-{val_part.date_id.max()}")

# targets and columns that are only known after the day ends must not be used as features
target_cols = ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]
feature_cols = [c for c in train_data.columns if c not in target_cols + ["date_id"]]

# drop features that are missing on more than a third of the training days (decided on the training part only)
features_delete = train_part[feature_cols].columns[train_part[feature_cols].isna().mean() > 0.33].tolist()
feature_cols = [c for c in feature_cols if c not in features_delete]
print(f"features: {len(feature_cols)}, dropped: {len(features_delete)}")

train days: 0-7237, validation days: 7238-9047
features: 82, dropped: 12


In [5]:
# missing values, scaling and the model are all fitted on the data passed to fit() only
def fit(df):
    means = df[feature_cols].mean()
    X = df[feature_cols].fillna(means)
    scaler = StandardScaler().fit(X)
    model = CatBoostRegressor(random_state=42, depth=1, verbose=0)
    model.fit(scaler.transform(X), df["forward_returns"])
    return means, scaler, model


def predict_returns(df, means, scaler, model):
    X = df.reindex(columns=feature_cols).fillna(means)
    return model.predict(scaler.transform(X))


# position between 0 and 2: 1 means fully invested, the prediction moves it up or down
def to_position(pred):
    return np.clip(1 + 50 * pred, 0, 2)


def evaluate(df, positions):
    solution = df[["forward_returns", "risk_free_rate"]].reset_index(drop=True)
    solution["row_id"] = np.arange(len(solution))
    submission = pd.DataFrame({"row_id": solution["row_id"], "prediction": positions})
    return score(solution, submission, row_id_column_name="row_id")


means, scaler, model = fit(train_part)

score_train = evaluate(train_part, to_position(predict_returns(train_part, means, scaler, model)))
score_val = evaluate(val_part, to_position(predict_returns(val_part, means, scaler, model)))
# reference: always fully invested, i.e. simply holding the S&P 500
score_hold = evaluate(val_part, np.ones(len(val_part)))

print("Score train:", np.round(score_train, 3))
print("Score validation:", np.round(score_val, 3))
print("Score validation, buy and hold:", np.round(score_hold, 3))

Score train: 0.816
Score validation: 0.563
Score validation, buy and hold: 0.624


In [6]:
# final model for the submission is refitted on all days
means, scaler, model = fit(train_data)


def predict(test_df: "pl.DataFrame") -> float:
    test_df = test_df.to_pandas()
    pred = predict_returns(test_df, means, scaler, model)[0]
    return float(to_position(pred))

In [7]:
sys.path.append(str(DATA_DIR))
import kaggle_evaluation.default_inference_server

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if __name__ == "__main__":
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        inference_server.serve()
    else:
        inference_server.run_local_gateway((str(DATA_DIR) + "/",))